In [0]:
import requests
import pandas as pd
from pyspark.sql.functions import *
from datetime import datetime
import json

#PARA PEGAR O ACESSO A ELETRICIDADE

def extrair_indicador_world_bank(indicator_code, indicator_name, nome_volume, data_inicio=None):
    url = f"http://api.worldbank.org/v2/country/all/indicator/{indicator_code}"
    params = {
    "format": "json",
    "per_page": 5000, #aqui, ele coloca a quantidade que vou pegar por paginas
    "page": 1   # inicia na página 1
    }
    wb_data = []

    if data_inicio:
        ano_inicio = str(data_inicio)[:4] # por ser por ano, e nao por mes, para garantir que pegue somente o ano
        params["date"] = f"{ano_inicio}:2026" 
    else:
        print("Realizando carregamento histórico completo...")

    while True:
        response = requests.get(url, params=params)
        response.raise_for_status()
        json_all = response.json()

        if len(json_all) < 2 or json_all[1] is None:
            break
    
        # o World Bank retorna o meta no índice 0 e os dados reais no índice 1
        meta = json_all[0]
        dados_reais = json_all[1]
    
        for item in dados_reais:
            wb_data.append({
                "indicator_code": item.get("indicator", {}).get("id"),
                "indicator_name": item.get("indicator", {}).get("value"),
                "country_name": item.get("country", {}).get("value"),
                "iso3": item.get("countryiso3code"),
                "date": item.get("date"),
                "metric_name": indicator_name,
                "metric": float(item.get("value")) if item.get("value") is not None else None 
            })            
        # verifica se a pagina atual já alcançou o total de paginas disponiveis
        if meta['page'] >= meta['pages']:
            break
        params['page'] += 1

    if not wb_data:
        print("Sem dados novos")
        return

    timestamp_extracao = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    pasta_destino = f"/Volumes/portfolio_energia/bronze/landing/{nome_volume}"
    os.makedirs(pasta_destino, exist_ok=True) # serve para criar um "subdiretório" dentro de um mesmo volume

    caminho_arquivo = f"{pasta_destino}/extracao_{timestamp_extracao}.json"
    with open(caminho_arquivo, "w", encoding="utf-8") as arquivo_json:
    json.dump(dados_extraidos, arquivo_json, ensure_ascii=False)
    #aqui, o with open abre um arquivo em branco a partir do caminho arquivo, e a instrução W(write),

def data_extracao(caminho_tabela):
    if spark.catalog.tableExists(caminho_tabela):
        return spark.sql(f"SELECT MAX(date) FROM {caminho_tabela}").collect()[0][0]
    else:   
        return None


data_pop = data_extracao("portfolio_energia.bronze.world_bank_bronze_populacao")
extrair_indicador_world_bank("SP.POP.TOTL", "Population", "landing_population", data_inicio=data_pop)

data_fdi = data_extracao("portfolio_energia.bronze.world_bank_bronze_fdi")
extrair_indicador_world_bank("BX.KLT.DINV.WD.GD.ZS", "FDI_Percent_GDP", "landing_fdi_percent_gdp", data_inicio=data_fdi)

data_elec = data_extracao("portfolio_energia.bronze.world_bank_bronze_eletricidade")
extrair_indicador_world_bank("EG.ELC.ACCS.ZS", "Electricity_Access", "landing_electricity_access", data_inicio=data_elec)

data_pib = data_extracao("portfolio_energia.bronze.world_bank_bronze_pib_per_capita")
extrair_indicador_world_bank("NY.GDP.PCAP.CD", "Pib_Per_Capita", "landing_pib_per_capita", data_inicio=data_pib)

In [0]:
def rodar_autoloader_world_bank(nome_volume, tabela_destino):
    caminho_base = f"/Volumes/portfolio_energia/bronze/landing/{nome_volume}"
    df_stream = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaLocation", f"{caminho_base}/_schema")
        .load(f"{caminho_base}/extracao_*.json")
    )
    df_stream_com_data = df_stream.withColumn("data_ingestao_bronze", current_timestamp())

    query = (
        df_stream_com_data.writeStream
        .format("delta")
        .option("checkpointLocation", f"{caminho_base}/_checkpoint")
        .trigger(availableNow=True) # Processa tudo o que chegou de novo e desliga
        .toTable(f"portfolio_energia.bronze.{tabela_destino}")
    )
    
    query.awaitTermination() # Espera o stream terminar de gravar antes de ir para a próxima linha do código

# Executando o Auto Loader para as duas tabelas Bronze:
rodar_autoloader_world_bank("landing_population", "world_bank_bronze_populacao")
rodar_autoloader_world_bank("landing_fdi_percent_gdp", "world_bank_bronze_fdi")
rodar_autoloader_world_bank("landing_electricity_access", "world_bank_bronze_eletricidade")
rodar_autoloader_world_bank("landing_pib_per_capita", "world_bank_bronze_pib_per_capita")